# DER Cross-Annotator Agreement Study (ChatGPT, Claude, Gemini)

A separate, lightweight notebook for comparing how consistently three LLMs apply your Domain Entity Recognition (DER) schema to a **50-sentence stratified subset** from InDiBias and IndiCasa. It does not train classifiers or run the main evaluation suite.

**Study design:** approximately 25 stereotype and 25 anti-stereotype examples, balanced across datasets where possible. All three annotators receive the same blinded sentence packet and the same frozen prompt. Agreement is exploratory and does not establish correctness; manually review disagreements.

## 1. Configure input files

Paths below point to `data/` in this repo (relative to wherever you run this notebook from — run it from the repo root, or adjust the paths). Each JSON should be a list of sentence strings or objects containing `sentence`, `text`, `input`, or `statement`. `DATA_FILES` maps (dataset, label) to file; pos=stereotype, neg=anti_stereotype.

In [ ]:
from pathlib import Path
import json, re, itertools, shutil
import numpy as np
import pandas as pd

# FIXED 2026-10-31: this dict had "stereotype" and "anti_stereotype" swapped relative to the
# convention used everywhere else in this project (pos=stereotype=1, neg=anti_stereotype=0,
# confirmed in src/data/der_graph.py's build_dataset and verified against real data). The three
# LLMs' annotations were already collected against the sample drawn under the OLD (wrong) mapping.
# Re-verified empirically: redrawing with the corrected mapping below picks an almost entirely
# different 50 sentences (only 2/50 overlap with the original draw) -- so DO NOT rerun this cell
# plus the sampling cell for the already-collected batch, or you invalidate the ChatGPT/Claude/
# Gemini annotations and have to re-query all three APIs. The agreement metrics themselves
# (pairwise_agreement_summary.csv etc.) never use class_label at all, so they are unaffected and
# did not need recomputing -- only sample_manifest.csv's class_label column needed a post-hoc
# relabel (done directly on that file, not via rerunning this cell). This corrected mapping is
# for any FUTURE from-scratch run only.
DATA_FILES = {
    ("InDiBias", "stereotype"): "data/indibias_pos.json",
    ("InDiBias", "anti_stereotype"): "data/indibias_neg.json",
    ("IndiCasa", "stereotype"): "data/indicasa_pos.json",
    ("IndiCasa", "anti_stereotype"): "data/indicasa_neg.json",
}
SAMPLE_SIZE, RANDOM_SEED = 50, 42
OUT = Path("outputs/der_cross_annotator")
OUT.mkdir(parents=True, exist_ok=True)

FROZEN_PACKET = OUT / "annotation_packet_blinded.json"
if FROZEN_PACKET.exists():
    print(f"⚠ {FROZEN_PACKET} already exists -- a sample has already been drawn and sent to the "
          f"three LLMs. Do NOT re-run the sampling cell (below) unless you intend to discard the "
          f"existing ChatGPT/Claude/Gemini annotations and re-query all three. If you only need to "
          f"relabel class_label, use the post-hoc relabel cell near the end of this notebook instead.")

for key, path in DATA_FILES.items(): print(key, path, Path(path).exists())

## 2. Load and normalize records

In [4]:
def records_from_json(obj):
    if isinstance(obj, list): return obj
    if isinstance(obj, dict):
        for k in ("data", "examples", "records", "sentences", "items"):
            if isinstance(obj.get(k), list): return obj[k]
    raise ValueError("Expected a JSON list or a dict containing a list under data/examples/records/sentences/items.")

def normalize(obj, dataset, label, path):
    out=[]
    for i, item in enumerate(records_from_json(obj)):
        if isinstance(item, str): sentence=item; meta={}
        elif isinstance(item, dict):
            sentence=next((item.get(k) for k in ("sentence","text","input","statement") if isinstance(item.get(k),str)), None)
            meta={k:v for k,v in item.items() if k not in ("annotations","der_annotations","tags")}
        else: continue
        if sentence and sentence.strip():
            out.append({"dataset":dataset,"class_label":label,"source_file":Path(path).name,"source_index":i,"sentence":sentence.strip(),"metadata":meta})
    return out

rows=[]
for (dataset,label), path in DATA_FILES.items():
    p=Path(path)
    if not p.exists(): print("SKIPPING missing path:",p); continue
    with p.open(encoding="utf-8") as f: obj=json.load(f)
    got=normalize(obj,dataset,label,path); print(dataset,label,len(got)); rows.extend(got)
if not rows: raise FileNotFoundError("No data loaded. Fix DATA_FILES paths in the first code cell.")
pool=pd.DataFrame(rows)
display(pool.groupby(["dataset","class_label"]).size().rename("n").reset_index())
display(pool[["dataset","class_label","sentence"]].head())

InDiBias stereotype 83
InDiBias anti_stereotype 243
IndiCasa stereotype 83
IndiCasa anti_stereotype 243


,dataset,class_label,n
0,InDiBias,anti_stereotype,243
1,InDiBias,stereotype,83
2,IndiCasa,anti_stereotype,243
3,IndiCasa,stereotype,83


,dataset,class_label,sentence
0,InDiBias,stereotype,Some people assume that being SC/ST means you'...
1,InDiBias,stereotype,The speaker argued that brahmin students does ...
2,InDiBias,stereotype,The brahmin politicians never deliver discrimi...
3,InDiBias,stereotype,Numerous vaishya entrepreneurs prioritize ethi...
4,InDiBias,stereotype,The contributions of leaders of brahmin commun...


## 3. Draw a stratified sample of 50

The target is 12–13 examples in each dataset × class stratum. Exact duplicate sentences are removed within a stratum. If any stratum is too small, the notebook fills from the remaining pool and reports the actual sample. If you have paired minimal examples with a reliable pair ID, adapt the sampler to keep pairs together.

In [ ]:
def stratified_sample(df, n=50, seed=42):
    rng=np.random.default_rng(seed)
    df=df.drop_duplicates(subset=["dataset","class_label","sentence"]).reset_index(drop=True)
    strata=list(itertools.product(["InDiBias","IndiCasa"],["stereotype","anti_stereotype"]))
    base,rem=divmod(n,len(strata)); targets={s:base+(i<rem) for i,s in enumerate(strata)}
    chosen=[]; shortages=[]
    for s,target in targets.items():
        cand=df.index[(df.dataset==s[0])&(df.class_label==s[1])].to_numpy()
        take=min(int(target),len(cand))
        if take: chosen.extend(rng.choice(cand,size=take,replace=False).tolist())
        if take<int(target): shortages.append((s,int(target),len(cand)))
    result=df.loc[chosen].copy()
    if len(result)<n:
        remain_df=df.drop(index=chosen,errors="ignore")
        take=min(n-len(result),len(remain_df))
        if take: result=pd.concat([result,remain_df.loc[rng.choice(remain_df.index.to_numpy(),size=take,replace=False)]])
    result=result.sample(frac=1,random_state=seed).reset_index(drop=True)
    result.insert(0,"sentence_id",[f"DER-{i:03d}" for i in range(1,len(result)+1)])
    return result,targets,shortages

_manifest_path = OUT / "sample_manifest.csv"
if _manifest_path.exists():
    # A sample was already drawn and sent to the three LLMs -- load it instead of redrawing, and
    # relabel class_label from source_file (pos=stereotype, neg=anti_stereotype) rather than trust
    # whatever class_label is already in the file, since that's exactly the column the original
    # bug corrupted. This does NOT change which sentences were sampled -- same sentence_ids, same
    # sentences, same already-collected model annotations stay valid.
    sample = pd.read_csv(_manifest_path)
    sample["class_label"] = sample["source_file"].apply(
        lambda f: "stereotype" if str(f).endswith("_pos.json") else "anti_stereotype")
    sample.to_csv(_manifest_path, index=False)
    targets, shortages = None, None
    print(f"Loaded existing frozen sample from {_manifest_path} ({len(sample)} sentences) and "
          f"relabeled class_label from source_file. Did NOT redraw -- see the warning in the first "
          f"code cell for why.")
else:
    sample,targets,shortages=stratified_sample(pool,SAMPLE_SIZE,RANDOM_SEED)
    print("Sample size:",len(sample),"| target counts:",targets,"| shortages:",shortages or "none")
    sample[["sentence_id","dataset","class_label","source_file","source_index","sentence"]].to_csv(_manifest_path,index=False)

display(sample.groupby(["dataset","class_label"]).size().rename("n").reset_index())
display(sample[["sentence_id","dataset","class_label","sentence"]].head(10))
# Hide dataset and class labels from annotators to reduce label priming.
packet=[{"sentence_id":r.sentence_id,"sentence":r.sentence} for r in sample.itertuples()]
(OUT/"annotation_packet_blinded.json").write_text(json.dumps(packet,ensure_ascii=False,indent=2),encoding="utf-8")
print("Saved blinded packet to",OUT)

## 4. Freeze your DER prompt and create the shared packet

Replace the placeholder below with your **final complete schema prompt**. Use exactly the same text for ChatGPT, Claude, and Gemini. Do not show existing DER annotations or labels to the annotators.

In [ ]:
# ⚠ UNRESOLVED as of 2026-10-31: this still reads "PASTE YOUR FINAL DER ANNOTATION PROMPT HERE" --
# the actual prompt sent to ChatGPT/Claude/Gemini for the already-collected batch was never saved
# anywhere. Without it, this study is not reproducible and a reviewer can reasonably ask for it.
# The assertion below stops this cell from silently writing a useless placeholder into
# shared_prompt_packet.json again. Fill in DER_PROMPT with the real prompt text used, then rerun.
DER_PROMPT = r"""PASTE YOUR FINAL DER ANNOTATION PROMPT HERE.

Return only a JSON array. Each item must contain sentence_id, sentence, and annotations.
Each annotation must contain span and node. For node = Identity Dimensions, label must be Religion, Caste, or Gender.
Allowed nodes: Identity Dimensions, Profession, Contextual Trigger, Situational Frame.
Use exact substrings from the sentence. Omit label for non-identity nodes unless your finalized schema says otherwise. Use an empty annotations list if no relevant spans exist.
"""
assert "PASTE YOUR FINAL" not in DER_PROMPT, (
    "DER_PROMPT is still the placeholder. The real prompt used for the already-collected "
    "ChatGPT/Claude/Gemini annotations was never recorded -- paste it in above before continuing.")
shared={"instructions":DER_PROMPT,"sentences":packet,"output_requirements":"Return a JSON array; preserve sentence_id and exact sentence text."}
(OUT/"shared_prompt_packet.json").write_text(json.dumps(shared,ensure_ascii=False,indent=2),encoding="utf-8")
print("Created",OUT/"shared_prompt_packet.json")
print("Run each LLM independently; save its JSON array as chatgpt_annotations.json, claude_annotations.json, gemini_annotations.json in",OUT)

## 5. Load the three model outputs

Save each model's JSON array in the output directory as `chatgpt_annotations.json`, `claude_annotations.json`, and `gemini_annotations.json`. Each record should look like:

```json
{"sentence_id":"DER-001","sentence":"exact sentence text","annotations":[{"span":"Brahmin","node":"Identity Dimensions","label":"Caste"}]}
```

Keep raw responses separately too. Do not let an annotator see another model's annotations before completing its own.

In [ ]:
import json
import re
from pathlib import Path

# --------------------------------------------------
# 1. Paths to the three annotation files
# --------------------------------------------------

MODEL_FILES = {
    "ChatGPT": "outputs/der_cross_annotator/chatgpt_annotations.json",
    "Claude": "outputs/der_cross_annotator/claude_annotations.json",
    "Gemini": "outputs/der_cross_annotator/gemini_annotations.json",
}


# --------------------------------------------------
# 2. Load and validate one model's JSON output
# --------------------------------------------------

def load_json_output(path):
    path = Path(path)  # Convert string to Path

    if not path.exists():
        raise FileNotFoundError(
            f"Missing file: {path}\n"
            "Check the dataset path in MODEL_FILES."
        )

    raw = path.read_text(encoding="utf-8").strip()

    # Remove optional Markdown code fences
    raw = re.sub(r"^\s*```(?:json)?\s*", "", raw, flags=re.I)
    raw = re.sub(r"\s*```\s*$", "", raw)

    try:
        obj = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Invalid JSON in {path.name}: {e}"
        ) from e

    # Accept a wrapper object containing an annotations list
    if isinstance(obj, dict) and isinstance(obj.get("annotations"), list):
        obj = obj["annotations"]

    if not isinstance(obj, list):
        raise ValueError(
            f"{path.name} must contain a JSON array of sentence records."
        )

    output = []

    for index, item in enumerate(obj):
        if not isinstance(item, dict):
            print(f"Warning: skipping invalid record {index} in {path.name}")
            continue

        sentence_id = normalize_id(item.get("sentence_id", ""))
        sentence = item.get("sentence", "")

        annotations = item.get("annotations", [])

        if not isinstance(annotations, list):
            print(
                f"Warning: annotations is not a list for "
                f"{sentence_id or f'record {index}'} in {path.name}"
            )
            annotations = []

        cleaned_annotations = []

        for annotation in annotations:
            if not isinstance(annotation, dict):
                continue

            span = annotation.get("span")
            node = annotation.get("node")

            if not isinstance(span, str) or not isinstance(node, str):
                continue

            cleaned_annotations.append({
                "span": span.strip(),
                "node": node.strip(),
                "label": str(annotation.get("label") or "").strip()
            })

        output.append({
            "model": None,
            "sentence_id": sentence_id,
            "sentence": sentence,
            "annotations": cleaned_annotations
        })

    return output


# --------------------------------------------------
# 3. Load all three models' annotations
# --------------------------------------------------

model_outputs = {}

# Check that the sampling/packet cell ran first
if "packet" not in globals():
    raise NameError(
        "The variable 'packet' is not defined. "
        "Run the cell that creates your sentence packet first."
    )

def normalize_id(sid):
    sid = str(sid).strip()
    match = re.fullmatch(r"DER-0*(\d+)", sid, flags=re.I)

    if match:
        return f"DER-{int(match.group(1)):02d}"

    return sid


expected_ids = {
    normalize_id(item.get("sentence_id", ""))
    for item in packet
    if isinstance(item, dict)
}

for model, filepath in MODEL_FILES.items():

    items = load_json_output(filepath)

    for item in items:
        item["model"] = model

    model_outputs[model] = items

    received_ids = {
        item["sentence_id"]
        for item in items
        if item["sentence_id"]
    }

    missing_ids = expected_ids - received_ids
    unexpected_ids = received_ids - expected_ids

    print(f"\n{model}")
    print("-" * 40)
    print("Records:", len(items))
    print("Missing IDs:", len(missing_ids))
    print("Unexpected IDs:", len(unexpected_ids))

    if missing_ids:
        print("Missing:", sorted(missing_ids))

    if unexpected_ids:
        print("Unexpected:", sorted(unexpected_ids))


# --------------------------------------------------
# 4. Final summary
# --------------------------------------------------

print("\nAll available annotation files have been loaded.")

for model, items in model_outputs.items():
    total_annotations = sum(
        len(item["annotations"]) for item in items
    )
    print(
        f"{model}: {len(items)} sentences, "
        f"{total_annotations} annotations"
    )

## 6. Check format and calculate pairwise agreement

- **Exact-span P/R/F1:** compares `(sentence_id, span)` sets, ignoring labels.
- **Typed annotation P/R/F1:** compares `(sentence_id, span, node, identity label)` sets.
- **Node agreement:** among exact spans emitted by both models, proportion with the same node.
- **Identity-label agreement:** among shared spans both models call `Identity Dimensions`, proportion with the same label.
- **Full annotation match:** proportion of sentence IDs whose entire normalized annotation sets match exactly.

These are descriptive pairwise metrics. They do not mean that the agreed annotation is correct.

In [14]:
VALID_NODES={"Identity Dimensions","Profession","Contextual Trigger","Situational Frame"}
VALID_LABELS={"Religion","Caste","Gender"}
long=[]
for model,items in model_outputs.items():
    for item in items:
        for a in item["annotations"]:
            long.append({"model":model,"sentence_id":item["sentence_id"],"sentence":item["sentence"],"span":a["span"],"node":a["node"],"label":a["label"],"valid_node":a["node"] in VALID_NODES,"valid_identity_label":a["node"]!="Identity Dimensions" or a["label"] in VALID_LABELS,"span_occurs":a["span"] in (item["sentence"] or "")})
long_df=pd.DataFrame(long)
if long_df.empty: raise ValueError("No annotations loaded. Check the three model JSON files.")
quality=long_df.groupby("model").agg(annotations=("span","size"),invalid_node=("valid_node",lambda x:int((~x).sum())),invalid_identity_label=("valid_identity_label",lambda x:int((~x).sum())),span_not_found=("span_occurs",lambda x:int((~x).sum()))).reset_index()
display(quality)
long_df.to_csv(OUT/"all_annotations_long.csv",index=False); quality.to_csv(OUT/"annotation_format_checks.csv",index=False)

def prf(a,b):
    inter=len(a&b); p=inter/len(a) if a else (1.0 if not b else 0.0); r=inter/len(b) if b else (1.0 if not a else 0.0); f=2*p*r/(p+r) if p+r else 0.0
    return p,r,f
def sets(items,typed=False):
    d={}
    for x in items:
        sid=x["sentence_id"]; d.setdefault(sid,set())
        for a in x["annotations"]:
            d[sid].add((a["span"],a["node"],a["label"]) if typed else a["span"])
    return d
def compare(A,B,ma,mb):
    sa,sb=sets(A),sets(B); ta,tb=sets(A,True),sets(B,True)
    spansA={(sid,s) for sid,vals in sa.items() for s in vals}; spansB={(sid,s) for sid,vals in sb.items() for s in vals}
    typedA={(sid,*v) for sid,vals in ta.items() for v in vals}; typedB={(sid,*v) for sid,vals in tb.items() for v in vals}
    sp=prf(spansA,spansB); tp=prf(typedA,typedB)
    mapA={(x["sentence_id"],a["span"]):(a["node"],a["label"]) for x in A for a in x["annotations"]}
    mapB={(x["sentence_id"],a["span"]):(a["node"],a["label"]) for x in B for a in x["annotations"]}
    common=set(mapA)&set(mapB); node_ag=sum(mapA[k][0]==mapB[k][0] for k in common)/len(common) if common else np.nan
    idkeys=[k for k in common if mapA[k][0]=="Identity Dimensions" and mapB[k][0]=="Identity Dimensions"]
    lab_ag=sum(mapA[k][1]==mapB[k][1] for k in idkeys)/len(idkeys) if idkeys else np.nan
    ids=set(sa)|set(sb); full=[sa.get(i,set())==sb.get(i,set()) and ta.get(i,set())==tb.get(i,set()) for i in ids]
    return {"model_a":ma,"model_b":mb,"span_precision":sp[0],"span_recall":sp[1],"span_f1":sp[2],"typed_precision":tp[0],"typed_recall":tp[1],"typed_f1":tp[2],"shared_exact_spans":len(common),"node_agreement_on_shared_spans":node_ag,"identity_label_agreement":lab_ag,"full_annotation_match_rate":float(np.mean(full)) if full else np.nan,"n_sentence_ids":len(ids)}

models=list(model_outputs); metrics=[]
for i in range(len(models)):
    for j in range(i+1,len(models)): metrics.append(compare(model_outputs[models[i]],model_outputs[models[j]],models[i],models[j]))
pairwise=pd.DataFrame(metrics); display(pairwise.round(3)); pairwise.to_csv(OUT/"pairwise_agreement_summary.csv",index=False)

,model,annotations,invalid_node,invalid_identity_label,span_not_found
0,ChatGPT,235,0,0,0
1,Claude,233,0,0,0
2,Gemini,229,0,0,0


,model_a,model_b,span_precision,span_recall,span_f1,typed_precision,typed_recall,typed_f1,shared_exact_spans,node_agreement_on_shared_spans,identity_label_agreement,full_annotation_match_rate,n_sentence_ids
0,ChatGPT,Claude,0.672,0.714,0.692,0.702,0.708,0.705,137,0.920,1.0,0.18,50
1,ChatGPT,Gemini,0.662,0.675,0.668,0.655,0.672,0.664,135,0.896,1.0,0.30,50
2,Claude,Gemini,0.766,0.735,0.750,0.730,0.742,0.736,147,0.891,1.0,0.24,50


## 7. Export span-level disagreements for adjudication

Review this file manually using your DER rules. A blank model column means that model did not emit the exact span. Do not automatically treat majority vote as ground truth; record the adjudication rationale separately.

In [ ]:
sentence_lookup={x["sentence_id"]:x["sentence"] for x in packet}
keys=sorted({(x["sentence_id"],a["span"]) for items in model_outputs.values() for x in items for a in x["annotations"]})
dis=[]
for sid,span in keys:
    row={"sentence_id":sid,"sentence":sentence_lookup.get(sid,""),"span":span}; vals=[]
    for model,items in model_outputs.items():
        found=[f"{a['node']}"+(f" [{a['label']}]" if a['label'] else "") for x in items if x["sentence_id"]==sid for a in x["annotations"] if a["span"]==span]
        row[model]="; ".join(found); vals.append(row[model])
    row["agreement_status"]="unanimous" if vals and vals[0] and all(v==vals[0] for v in vals) else ("omitted_by_all" if all(not v for v in vals) else "disagreement")
    dis.append(row)
dis_df=pd.DataFrame(dis); display(dis_df[dis_df.agreement_status=="disagreement"].head(30)); dis_df.to_csv(OUT/"span_disagreements_for_adjudication.csv",index=False)

# Write a compact report and zip all files currently in the output directory.
report=["DER Cross-Annotator Agreement Study",f"Sample size: {len(packet)}",f"Random seed: {RANDOM_SEED}","", "Sample counts:",sample.groupby(["dataset","class_label"]).size().to_string(),"", "Format checks:",quality.to_string(index=False),"", "Pairwise agreement:",pairwise.round(3).to_string(index=False)]
(OUT/"cross_annotator_report.txt").write_text("\n".join(report),encoding="utf-8")
zip_path=shutil.make_archive(str(OUT.parent/"der_cross_annotator_outputs"),"zip",OUT)
print("ZIP:",zip_path,"\nOutput files:")
for p in sorted(OUT.iterdir()): print(" -",p.name)